# This notebook is to combine data from several years to facilitate the model training

In [1]:
import pandas as pd                            
import os
import re
from collections import defaultdict

In [2]:
def all_files_in(data_path):
    """
    Collects the paths of all files in the specified directory and its subdirectories.

    Args:
        data_path (str): The path of the directory to search for files.

    Returns:
        list: A list containing the full paths of all files found within the directory.
        
    Raises:
        ValueError: If the specified path is not a directory.
    """
    if not os.path.isdir(data_path):
        raise ValueError(f"The specified path '{data_path}' is not a valid directory.")
    
    all_files = []
    for dir_path, _, file_names in os.walk(data_path):
        for file_name in file_names:
            all_files.append(os.path.join(dir_path, file_name))
    
    return all_files

In [3]:
def categorize_files(file_list, location):
    categorized_files = defaultdict(list)

    # Build regex patterns dynamically using the location
    patterns = {
        'input_t0': rf"input-{location}-t0-(\d{{4}})\.csv$",        
        't0': rf"output-{location}-t0-(\d{{4}})\.csv$",
        't1': rf"output-{location}-t1-(\d{{4}})\.csv$",
        't2': rf"output-{location}-t2-(\d{{4}})\.csv$",
        't4': rf"output-{location}-t4-(\d{{4}})\.csv$",
        't6': rf"output-{location}-t6-(\d{{4}})\.csv$",
    }

    for file in file_list:
        filename = os.path.basename(file)
        for category, pattern in patterns.items():
            if re.match(pattern, filename):
                categorized_files[category].append(file)

    return (
        categorized_files['input_t0'],
        categorized_files['t0'],
        categorized_files['t1'],
        categorized_files['t2'],
        categorized_files['t4'],
        categorized_files['t6']
    )


In [4]:
def merge_csv_from_list(csv_files, output_filename, keep_header=True):
    """
    Merges a list of CSV files, keeps the order, and handles headers based on user input.
    The merged file will be saved directly.

    Args:
    - csv_files: List of CSV file paths to merge
    - keep_header: Whether to keep the header (True to keep the header, False to ignore it)
    - output_filename: The filename to save the merged CSV (default is 'merged_output.csv')
    """
    merged_df = pd.DataFrame()

    for i, file in enumerate(csv_files):
        # Read each CSV file
        if i == 0 or keep_header:  # Keep header for the first file or if keep_header is True
            df = pd.read_csv(file)
        else:  # Skip header for subsequent files if keep_header is False
            df = pd.read_csv(file, header=None)
            # Skip the header row (if any) by renaming columns to match the first file
            df.columns = merged_df.columns

        # Merge the data
        merged_df = pd.concat([merged_df, df], ignore_index=True)

    # Save the merged DataFrame directly to a CSV
    merged_df.to_csv(output_filename, index=False)

In [5]:
def merge_binary_csv(csv_files, output_filename):
    """
    Merges a list of CSV files containing flattened binary matrices, without adding headers.
    The merged file will be saved directly without any headers.

    Args:
    - csv_files: List of CSV file paths to merge
    - output_filename: The filename to save the merged CSV (default is 'merged_binary_matrix.csv')
    """
    merged_df = pd.DataFrame()

    for i, file in enumerate(csv_files):
        # Read the CSV without headers (since it's a flattened binary matrix)
        df = pd.read_csv(file, header=None)

        # Concatenate the current file's data to the merged dataframe
        merged_df = pd.concat([merged_df, df], ignore_index=True)

    # Save the merged dataframe directly to a CSV, without index or header
    merged_df.to_csv(output_filename, index=False, header=False)

In [6]:
def clean_and_deduplicate(df):
    """
    Cleans a DataFrame by:
    - Removing duplicate rows based on ['year', 'month', 'day', 'hour', 'minute']
    - Removing rows where 'year' column literally contains the string 'year'
    - Casting time-related columns to integers
    - Resetting the index
    
    Args:
        df (pd.DataFrame): Input DataFrame with possible header row as a data row
    
    Returns:
        pd.DataFrame: Cleaned DataFrame
    """
    # Drop duplicates based on timestamp fields
    df = df.drop_duplicates(subset=['year', 'month', 'day', 'hour', 'minute'])
    
    # Remove duplicated header row (if exists)
    df = df[df['year'] != 'year']
    
    # Convert time columns to integer
    df = df.astype({
        'year': int,
        'month': int,
        'day': int,
        'hour': int,
        'minute': int
    })

    # Reset index
    df = df.reset_index(drop=True)

    return df

In [7]:
location="zambia"

In [11]:
data_type = "train"
parent_folder = f'/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Data/Zambia/64x64/{data_type}'

all_files = all_files_in(parent_folder)
input_t0, files_map_t0, files_map_t1, files_map_t2, files_map_t4, files_map_t6 = categorize_files(all_files, location=location)

input_t0.sort()
files_map_t0.sort()
files_map_t1.sort()
files_map_t2.sort()
files_map_t4.sort()
files_map_t6.sort()

In [12]:
output_folder = '/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Data/Zambia/64x64/full'

In [13]:
merge_csv_from_list(input_t0, f'{output_folder}/{data_type}-data-{location}-input-t0.csv')
merge_binary_csv(files_map_t0, f'{output_folder}/{data_type}-data-{location}-output-map-t0.csv')
merge_binary_csv(files_map_t1, f'{output_folder}/{data_type}-data-{location}-output-map-t1.csv')
merge_binary_csv(files_map_t2, f'{output_folder}/{data_type}-data-{location}-output-map-t2.csv')
merge_binary_csv(files_map_t4, f'{output_folder}/{data_type}-data-{location}-output-map-t4.csv')
merge_binary_csv(files_map_t6, f'{output_folder}/{data_type}-data-{location}-output-map-t6.csv')